# NCBI Entrez Fetch Examples

This notebook demonstrates direct NCBI Entrez operations using **`run_ncbi_fetch`**.

- **esearch** -- Search for IDs by query term
- **esummary** -- Retrieve record summaries by ID
- **efetch** -- Fetch sequences in FASTA or other formats

## Imports

In [ ]:
from bio_programming_tools.tools.database_retrieval import (
    run_ncbi_fetch,
    NCBIFetchInput,
    NCBIFetchConfig,
)

## API Reference

**`NCBIFetchInput`**

| Field | Type | Default | Description |
|---|---|---|---|
| `db` | `Literal["protein", "nuccore", "gene", "nucleotide"]` | *(required)* | NCBI database to query |
| `operation` | `Literal["esearch", "esummary", "efetch"]` | *(required)* | Entrez operation |
| `identifier` | `Optional[str]` | `None` | Accession or ID (required for efetch/esummary) |
| `search_term` | `Optional[str]` | `None` | Entrez query (required for esearch) |
| `rettype` | `str` | `"fasta"` | Return type for efetch |
| `seq_start` | `Optional[int]` | `None` | 1-indexed inclusive start for subsequence |
| `seq_stop` | `Optional[int]` | `None` | 1-indexed inclusive stop for subsequence |
| `strand` | `Optional[Literal["+", "-"]]` | `None` | Strand for nucleotide retrieval |
| `max_results` | `int` | `5` | Maximum results from esearch |

**`NCBIFetchConfig`**

| Field | Type | Default | Description |
|---|---|---|---|
| `request_timeout_seconds` | `int` | `15` | HTTP timeout |
| `http_retries` | `int` | `2` | Retries for HTTP requests |
| `backoff_seconds` | `float` | `1.0` | Base exponential backoff |
| `ncbi_api_key` | `Optional[str]` | `None` | Optional NCBI API key |
| `ncbi_email` | `Optional[str]` | `None` | Optional NCBI email |

**`NCBIFetchOutput`**

| Field | Type | Description |
|---|---|---|
| `operation` | `str` | Entrez operation performed |
| `db` | `str` | Database queried |
| `ids` | `List[str]` | esearch ID list |
| `summary` | `Dict[str, Any]` | esummary result data |
| `fasta_records` | `List[NCBIFastaRecord]` | Parsed FASTA records (efetch) |
| `raw_text` | `Optional[str]` | Raw text for non-FASTA rettypes |
| `source_url` | `Optional[str]` | Sanitized request URL |

## 1. Search for Protein IDs (esearch)

Search the NCBI protein database for lac repressor in E. coli.

In [ ]:
inputs = NCBIFetchInput(
    db="protein",
    operation="esearch",
    search_term='"lacI"[Gene] AND "Escherichia coli"[Organism]',
    max_results=5,
)

output = run_ncbi_fetch(inputs, NCBIFetchConfig())

print(f"Operation: {output.operation}")
print(f"Database: {output.db}")
print(f"IDs found: {output.ids}")

## 2. Fetch a Protein FASTA (efetch)

Fetch the TP53 protein sequence from NCBI by RefSeq accession.

In [ ]:
inputs = NCBIFetchInput(
    db="protein",
    operation="efetch",
    identifier="NP_000537.3",
    rettype="fasta",
)

output = run_ncbi_fetch(inputs, NCBIFetchConfig())

record = output.fasta_records[0]
print(f"Accession: {record.accession}")
print(f"Header: {record.header[:80]}...")
print(f"Sequence length: {len(record.sequence)}")
print(f"Preview: {record.sequence[:60]}...")

## 3. Fetch a Genomic Subsequence

Retrieve a specific region of the E. coli K-12 genome using coordinates and strand.

In [ ]:
# Fetch a 200bp region from the lacI locus (minus strand)
inputs = NCBIFetchInput(
    db="nuccore",
    operation="efetch",
    identifier="NC_000913.3",
    rettype="fasta",
    seq_start=366428,
    seq_stop=366628,
    strand="-",
)

output = run_ncbi_fetch(inputs, NCBIFetchConfig())

record = output.fasta_records[0]
print(f"Accession: {record.accession}")
print(f"Sequence length: {len(record.sequence)}")
print(f"Preview: {record.sequence[:80]}...")
print(f"Source URL: {output.source_url}")

## 4. Retrieve a Gene Summary (esummary)

Get summary metadata for a gene record, including genomic coordinates.

In [ ]:
# First, find the gene ID for TP53 in humans
search_output = run_ncbi_fetch(
    NCBIFetchInput(
        db="gene",
        operation="esearch",
        search_term='"TP53"[Gene Name] AND "Homo sapiens"[Organism]',
        max_results=1,
    ),
    NCBIFetchConfig(),
)
gene_id = search_output.ids[0]
print(f"Gene ID: {gene_id}")

# Now get the gene summary
summary_output = run_ncbi_fetch(
    NCBIFetchInput(
        db="gene",
        operation="esummary",
        identifier=gene_id,
    ),
    NCBIFetchConfig(),
)

gene_data = summary_output.summary.get(gene_id, {})
print(f"Name: {gene_data.get('name')}")
print(f"Description: {gene_data.get('description')}")
print(f"Organism: {gene_data.get('organism', {}).get('scientificname')}")

genomic_info = gene_data.get("genomicinfo", [])
if genomic_info:
    region = genomic_info[0]
    print(f"Chromosome: {region.get('chraccver')}")
    print(f"Coordinates: {region.get('chrstart')}-{region.get('chrstop')}")